# Verifikasi Skor di Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/galiihajiip/agridata-telepati/blob/main/notebooks/verify_colab.ipynb)

Tim **Di Bandung Di Ganesha**, Universitas Pembangunan Nasional "Veteran" Jawa Timur.

Notebook ini hanya untuk **memverifikasi angka yang kami laporkan**, tanpa pelatihan ulang.
Laporan lengkap ada pada `notebooks/final_agriData_telepati8.ipynb`.

Langkah yang dijalankan:

1. *Clone* repository dan memasang `ultralytics==8.4.154`.
2. Mengunduh bobot final dari GitHub Release dan memverifikasi SHA-256-nya.
3. Menyiapkan dataset resmi dengan `scripts/prepare_dataset.py`, sama persis dengan pipeline kami.
4. Mengevaluasi dengan `scripts/evaluate.py` (NMS IoU 0,5, resolusi 640) lalu membaca dua angka
   leaderboard dengan `scripts/compute_official_metrics.py`.
5. Membandingkan hasilnya dengan angka yang kami laporkan, dengan toleransi 2 persen relatif.

Dataset resmi lomba tidak kami sertakan di repository. Unggah zip dataset resmi (berisi folder
`train/`, `valid/`, `test/` dengan `_annotations.coco.json`) ke Colab atau Google Drive, lalu isi
`DATASET_ZIP` di sel berikut. Pilih runtime GPU (Runtime > Change runtime type) agar lebih cepat;
CPU juga dapat dipakai.

**Cara pakai:** isi `DATASET_ZIP`, lalu Runtime > Run all.

In [ ]:
# KONFIGURASI: hanya DATASET_ZIP yang perlu diisi.
DATASET_ZIP = "/content/dataset_telepati.zip"   # zip dataset resmi, atau folder hasil ekstraknya
SPLITS = ["valid"]                              # tambahkan "test" untuk memverifikasi split test

REPO_URL = "https://github.com/galiihajiip/agridata-telepati.git"
WEIGHTS_URL = "https://github.com/galiihajiip/agridata-telepati/releases/download/v1.0.0-final-model/best.pt"
WEIGHTS_SHA256 = "4dc86ac0e53d2849fb1b79f1a68dcb5284ae6f1e8fef27e4416fa54c6b772a9f"

# Angka yang kami laporkan (artifacts/reports/official_metrics_{split}.json), dalam persen.
REPORTED = {"valid": {"mAP50": 70.71, "f1": 72.33}, "test": {"mAP50": 70.65, "f1": 71.45}}
TOLERANCE_RELATIVE = 0.02

In [ ]:
import os, sys, subprocess, hashlib, zipfile, json, urllib.request
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
WORK = Path("/content") if IN_COLAB else Path(os.environ.get("VERIFY_WORKDIR", "/tmp/agridata_verify"))
WORK.mkdir(parents=True, exist_ok=True)

if IN_COLAB:
    repo = WORK / "agridata-telepati"
    if not repo.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(repo)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.154"], check=True)
else:
    # Uji lokal: notebook dijalankan dari dalam repository.
    repo = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.chdir(repo)
os.environ["YOLO_OFFLINE"] = "1"
print("Repository :", repo)
print("Commit     :", subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip())

In [ ]:
def find_dataset_root(base: Path) -> Path:
    for ann in sorted(base.rglob("_annotations.coco.json")):
        root = ann.parent.parent
        if all((root / s / "_annotations.coco.json").exists() for s in ("train", "valid", "test")):
            return root
    raise FileNotFoundError(f"Tidak menemukan folder train/valid/test berisi _annotations.coco.json di {base}")

src = Path(os.environ.get("VERIFY_DATASET", DATASET_ZIP))
if src.suffix == ".zip":
    extract_dir = WORK / "dataset_raw"
    if not extract_dir.exists():
        with zipfile.ZipFile(src) as z:
            z.extractall(extract_dir)
    src = extract_dir
DATASET_ROOT = find_dataset_root(src)
print("Dataset resmi:", DATASET_ROOT)

In [ ]:
weights = WORK / "best.pt"
if not weights.exists():
    urllib.request.urlretrieve(WEIGHTS_URL, weights)
h = hashlib.sha256()
with open(weights, "rb") as f:
    for chunk in iter(lambda: f.read(1 << 20), b""):
        h.update(chunk)
print("SHA-256 unduhan:", h.hexdigest())
assert h.hexdigest() == WEIGHTS_SHA256, "SHA-256 tidak cocok dengan bobot yang kami laporkan."
print("SHA-256 cocok dengan bobot final.")

In [ ]:
PREPARED = WORK / "prepared"
OUT = WORK / "verify_reports"
OUT.mkdir(exist_ok=True)

def run(cmd):
    print("$", " ".join(str(c) for c in cmd))
    r = subprocess.run([str(c) for c in cmd], capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:])
        raise RuntimeError(f"Perintah gagal dengan kode {r.returncode}")

if not (PREPARED / "data.yaml").exists():
    run([sys.executable, "scripts/prepare_dataset.py", "--dataset-root", DATASET_ROOT,
         "--output-dir", PREPARED, "--summary-dir", OUT, "--seed", "42"])
print("Data siap:", PREPARED)

In [ ]:
results = {}
for split in SPLITS:
    run([sys.executable, "scripts/evaluate.py", "--weights", weights, "--split", split,
         "--prepared-dir", PREPARED, "--nms-iou", "0.5",
         "--report-dir", OUT, "--predictions-dir", OUT / "predictions", "--figures-dir", OUT / "figures"])
    run([sys.executable, "scripts/compute_official_metrics.py", "--split", split, "--report-dir", OUT])
    with open(OUT / f"official_metrics_{split}.json") as f:
        results[split] = json.load(f)["metrik_dilaporkan"]

print(f"{'Split':6s} {'Metrik':8s} {'Dilaporkan':>11s} {'Hasil ini':>10s} {'Selisih relatif':>16s}  Status")
all_ok = True
for split, got in results.items():
    for key, label in (("mAP50_persen", "mAP50"), ("f1_persen", "f1")):
        ref = REPORTED[split][label]
        rel = abs(got[key] - ref) / ref
        ok = rel <= TOLERANCE_RELATIVE
        all_ok &= ok
        print(f"{split:6s} {label:8s} {ref:10.2f}% {got[key]:9.2f}% {rel * 100:15.2f}%  {'LULUS' if ok else 'GAGAL'}")
print("\nKESIMPULAN:", "seluruh angka dalam toleransi 2 persen" if all_ok else "ada angka di luar toleransi")